# Context Engineering for AI Coding Agent Infrastructure

## What you will build

You will build the code that decides what a coding agent reads before it touches a monorepo. The
repository holds a React frontend and an Express backend. One instruction file carries the build
settings for the whole tree, and a rule file for the backend adds database safety checks that only
matter when backend code changes.

The model only knows what that code puts in front of it. The diagram shows the four mistakes this
course stops: backend rules paid for on every frontend task, the database rule missing on a backend
change, one instruction written into several files, and rules lost when a long session is
summarised.

![What you will build](images/context-overview.svg)

## Step 0: Set up the client and the model

Every call in this notebook goes through the repository's own client. Without an API key it replays
responses recorded from real runs, so you can follow the whole course for free, and with a key it
calls the model live.

In [1]:
import fnmatch
import pathlib
import re
import shutil
import tempfile

from vault import get_client, load_env, model_for, provider_truth

load_env()
client = get_client("04-context-engineering/01-assemble-agent-context")
MODEL = model_for("default")
SUMMARY_MODEL = model_for("small")

print(f"Client ready. The coding agent uses {MODEL}, and summaries use {SUMMARY_MODEL}.")

Client ready. The coding agent uses google/gemini-2.5-flash-lite, and summaries use mistralai/mistral-nemo.


## Step 1: Lay out the monorepo and its instruction file

The agent works on a real folder, so we start by writing a small monorepo to a temporary directory.
The **instruction file**, `AGENTS.md` at the root, holds the settings that apply to every task in
the repository, whichever folder the task touches.

![Lay out the monorepo and its instruction file](images/rule-selection-step-1.svg)

In [2]:
WORKSPACE = pathlib.Path(tempfile.mkdtemp(prefix="shop-monorepo-"))
SOURCE_FILES = ["src/ui/App.tsx", "src/ui/components/OrderList.tsx", "src/ui/kit/Button.tsx",
                "src/api/server.ts", "src/api/db.ts", "src/api/routes/orders.ts"]
for relative_path in SOURCE_FILES:
    (WORKSPACE / relative_path).parent.mkdir(parents=True, exist_ok=True)
    (WORKSPACE / relative_path).touch()

INSTRUCTION_PATH = WORKSPACE / "AGENTS.md"
INSTRUCTION_PATH.write_text("""# Build settings for the shop monorepo
- Use pnpm for every package command, never npm or yarn.
- TypeScript runs in strict mode, so never add an any type.
- Run pnpm test before you commit.
- The React frontend lives in src/ui and the Express backend lives in src/api.
""")
print(f"{len(SOURCE_FILES)} source files and AGENTS.md written to a temporary folder")

6 source files and AGENTS.md written to a temporary folder


A **path-scoped rule** is a rule file that says which paths it applies to, in a `paths:` line at
its top. That line holds a **glob**, a file pattern where `*` stands for any name and `**` stands for
any number of folders. Each rule below was written by a different team, and each copied a few build
lines from the instruction file so that it would read as complete on its own.

In [3]:
RULE_FILES = {
    "api-database.md": """---
paths: src/api/**/*.ts
---
# Database safety for backend code
- Use pnpm for every package command, never npm or yarn.
- Run pnpm test before you commit.
- Before you change any file, run pnpm db:check to confirm the database is reachable.
- Send every SQL query through db.query(sql, params) with $1 placeholders.
- Run any write that touches more than one table inside db.transaction.
""",
    "ui-components.md": """---
paths: src/ui/**/*.tsx
---
# React components
- Use pnpm for every package command, never npm or yarn.
- Build every component from the shared parts in src/ui/kit.
- Give every button and input an aria-label.
""",
}
(WORKSPACE / "rules").mkdir()
for name, text in RULE_FILES.items():
    (WORKSPACE / "rules" / name).write_text(text)
    print(f"rules/{name}  {len(text)} characters")

rules/api-database.md  391 characters
rules/ui-components.md  213 characters


## Step 2: Send every rule on every task and measure the cost

The simplest way to build the context is to read the instruction file and every rule file, join
them, and send the lot with every task. We try that first, on a task that only touches the
frontend, to see what the backend rule does there.

In [4]:
def load_every_rule():
    """The first version: the instruction file and every rule file, on every task."""
    files = [INSTRUCTION_PATH, *sorted((WORKSPACE / "rules").glob("*.md"))]
    return "\n\n".join(path.read_text() for path in files)


def ask_coding_agent(context, task):
    """One turn of the coding agent. Returns the answer and the prompt tokens charged."""
    response = client.chat.completions.create(
        model=MODEL, max_tokens=400,
        messages=[{"role": "system", "content": context},
                  {"role": "user", "content": task}])
    return response.choices[0].message.content, response.usage.prompt_tokens

The task changes one React component, so a database check has no place in the plan. The next cell
asks five times and counts the plans that run `pnpm db:check` anyway. Each attempt also prints what
the request cost, where a **token** is the unit a model reads and bills in, roughly a short word or
part of one.

In [5]:
UI_TASK = ("Changed file: src/ui/components/OrderList.tsx\n"
           "Add a loading spinner to the order list while the orders load. "
           "List the steps you will take.")

every_rule = load_every_rule()
stray_checks = 0
for attempt in range(1, 6):
    answer, prompt_tokens = ask_coding_agent(every_rule, UI_TASK)
    stray_checks += "db:check" in answer
    print(f"attempt {attempt}: {prompt_tokens} prompt tokens, "
          f"plan runs the database check: {'db:check' in answer}")
print(f"\n{stray_checks} of 5 frontend plans ran a backend database check")

attempt 1: 264 prompt tokens, plan runs the database check: False


attempt 2: 264 prompt tokens, plan runs the database check: False


attempt 3: 264 prompt tokens, plan runs the database check: False


attempt 4: 264 prompt tokens, plan runs the database check: False


attempt 5: 264 prompt tokens, plan runs the database check: False

0 of 5 frontend plans ran a backend database check


The model left the database check out of all five plans, so this time the backend rule did no
visible harm to the answers. It still cost money on every attempt, because each request paid for
264 prompt tokens, and the next step shows how many of them were rules that never applied to a
React component.

## Step 3: Read each rule's glob from its frontmatter

Each rule file already says where it applies, so the fix starts by reading that line instead of
throwing it away. `read_rule_file` splits a file into its glob and its body, and it refuses a rule
with no `paths:` line, because a rule with no scope would quietly load on every task again.

![Read each rule's glob from its frontmatter](images/rule-selection-step-2.svg)

In [6]:
def read_rule_file(path):
    """Split a rule file into the glob in its frontmatter and the body under it."""
    text = path.read_text()
    if not text.startswith("---\n"):
        raise ValueError(f"{path.name} has no frontmatter, so nothing says where it applies")
    frontmatter, body = text[len("---\n"):].split("---\n", 1)
    if not frontmatter.startswith("paths:"):
        raise ValueError(f"{path.name} has frontmatter but no paths: line")
    glob = frontmatter.removeprefix("paths:").strip()
    return {"name": path.name, "glob": glob, "body": body.strip()}


def read_all_rules():
    """Every rule file in the rules folder, read fresh from disk."""
    return [read_rule_file(path) for path in sorted((WORKSPACE / "rules").glob("*.md"))]


RULES = read_all_rules()
for rule in RULES:
    print(f"{rule['name']:18} applies to {rule['glob']}")

api-database.md    applies to src/api/**/*.ts
ui-components.md   applies to src/ui/**/*.tsx


## Step 4: Select only the rules whose glob matches a changed path

The context for a task is now the instruction file plus the rules that match at least one path the
task changed. The matching function is a parameter, and we start with Python's own `fnmatch`,
which is what most first versions reach for.

![Select only the rules whose glob matches a changed path](images/rule-selection-step-3.svg)

In [7]:
def assemble_scoped_context(changed_paths, match_path):
    """The instruction file, then the body of every rule whose glob matches a changed path."""
    selected = [rule for rule in RULES
                if any(match_path(path, rule["glob"]) for path in changed_paths)]
    context = "\n\n".join([INSTRUCTION_PATH.read_text().strip()]
                          + [rule["body"] for rule in selected])
    return context, [rule["name"] for rule in selected]


ui_context, selected = assemble_scoped_context(["src/ui/components/OrderList.tsx"],
                                               fnmatch.fnmatch)
print(f"rules selected for the frontend task: {selected}")

rules selected for the frontend task: ['ui-components.md']


Only the frontend rule matched, so the same task goes out five more times with the scoped context.

In [8]:
scoped_stray_checks = 0
for attempt in range(1, 6):
    answer, scoped_tokens = ask_coding_agent(ui_context, UI_TASK)
    scoped_stray_checks += "db:check" in answer
    print(f"attempt {attempt}: {scoped_tokens} prompt tokens, "
          f"plan runs the database check: {'db:check' in answer}")
print(f"\nevery rule : {stray_checks} of 5 plans ran the database check, {prompt_tokens} tokens")
print(f"scoped     : {scoped_stray_checks} of 5 plans ran the database check, {scoped_tokens} tokens")

attempt 1: 150 prompt tokens, plan runs the database check: False


attempt 2: 150 prompt tokens, plan runs the database check: False


attempt 3: 150 prompt tokens, plan runs the database check: False


attempt 4: 152 prompt tokens, plan runs the database check: False


attempt 5: 150 prompt tokens, plan runs the database check: False

every rule : 0 of 5 plans ran the database check, 264 tokens
scoped     : 0 of 5 plans ran the database check, 150 tokens


Neither version put a database check in a frontend plan, but the scoped context cost about 150
prompt tokens against 264. Every frontend task now pays for 114 fewer tokens of rules that never
applied to it, and that saving repeats on every frontend task the agent runs.

## Step 5: Match globs across folders the way the rule author meant

The frontend task works, so we give the agent a backend task on `src/api/db.ts`, the file that talks
to the database. The database rule asks for `$1` placeholders and a transaction around any write
to two tables, and the next cell checks both in every answer.

In [9]:
BACKEND_TASK = ("Changed file: src/api/db.ts\n"
                "Add a function deleteCustomer(customerId) that deletes the customer's "
                "orders and then the customer. Show only the code.")


def wraps_writes_in_transaction(answer):
    """True when the code runs its writes in a transaction, by helper or by hand."""
    return "db.transaction" in answer or "BEGIN" in answer


backend_context, selected = assemble_scoped_context(["src/api/db.ts"], fnmatch.fnmatch)
print(f"rules selected for src/api/db.ts: {selected}\n")
for attempt in range(1, 4):
    answer, _ = ask_coding_agent(backend_context, BACKEND_TASK)
    print(f"attempt {attempt}: transaction: {wraps_writes_in_transaction(answer)}, "
          f"$1 placeholders: {'$1' in answer}")

rules selected for src/api/db.ts: []



attempt 1: transaction: False, $1 placeholders: False


attempt 2: transaction: True, $1 placeholders: True


attempt 3: transaction: False, $1 placeholders: False


No rule was selected, so the agent guessed at a data layer it had never been shown. Only one of
the three answers used a transaction and `$1` placeholders, and the other two deleted the orders and
the customer as two separate writes. The cause is in the matcher rather than the model, and the next
cell shows it on three backend paths.

In [10]:
for path in ["src/api/db.ts", "src/api/server.ts", "src/api/routes/orders.ts"]:
    print(f"{path:26} matches src/api/**/*.ts: {fnmatch.fnmatch(path, 'src/api/**/*.ts')}")

src/api/db.ts              matches src/api/**/*.ts: False
src/api/server.ts          matches src/api/**/*.ts: False
src/api/routes/orders.ts   matches src/api/**/*.ts: True


`fnmatch` reads `**` as two ordinary stars, and its star matches any characters, slashes included.
The slash after `**` stays a literal slash, so the glob only matches a file with at least one folder
between `src/api` and its name. The rule author meant `**/` as any number of folders, including
none, which is how shells and coding tools read it.

In [11]:
def translate_glob_to_regex(glob):
    """`**/` matches any number of folders, `*` and `?` stay inside one folder."""
    pattern, i = "", 0
    while i < len(glob):
        if glob.startswith("**/", i):
            pattern, i = pattern + "(?:[^/]+/)*", i + 3
        elif glob.startswith("**", i):
            pattern, i = pattern + ".*", i + 2
        elif glob[i] in "*?":
            pattern, i = pattern + ("[^/]*" if glob[i] == "*" else "[^/]"), i + 1
        else:
            pattern, i = pattern + re.escape(glob[i]), i + 1
    return re.compile(pattern + r"\Z")


def match_path_to_glob(path, glob):
    """True when the whole path matches the glob, folder by folder."""
    return translate_glob_to_regex(glob).match(path) is not None

The same three paths go through both matchers, with one frontend case added where `fnmatch` is too
loose rather than too strict.

In [12]:
GLOB_CASES = [("src/api/db.ts", "src/api/**/*.ts"),
              ("src/api/server.ts", "src/api/**/*.ts"),
              ("src/api/routes/orders.ts", "src/api/**/*.ts"),
              ("src/ui/components/OrderList.tsx", "src/ui/*.tsx")]
print(f"{'path':32} {'glob':18} fnmatch  folder-aware")
for path, glob in GLOB_CASES:
    print(f"{path:32} {glob:18} {str(fnmatch.fnmatch(path, glob)):8} "
          f"{match_path_to_glob(path, glob)}")

path                             glob               fnmatch  folder-aware
src/api/db.ts                    src/api/**/*.ts    False    True
src/api/server.ts                src/api/**/*.ts    False    True
src/api/routes/orders.ts         src/api/**/*.ts    True     True
src/ui/components/OrderList.tsx  src/ui/*.tsx       True     False


With the folder-aware matcher, the backend task gets the database rule, and the same request goes
out three more times.

In [13]:
backend_context, selected = assemble_scoped_context(["src/api/db.ts"], match_path_to_glob)
print(f"rules selected for src/api/db.ts: {selected}\n")
for attempt in range(1, 4):
    answer, _ = ask_coding_agent(backend_context, BACKEND_TASK)
    print(f"attempt {attempt}: transaction: {wraps_writes_in_transaction(answer)}, "
          f"$1 placeholders: {'$1' in answer}, calls db.transaction: {'db.transaction' in answer}")

rules selected for src/api/db.ts: ['api-database.md']



attempt 1: transaction: True, $1 placeholders: True, calls db.transaction: False


attempt 2: transaction: True, $1 placeholders: True, calls db.transaction: False


attempt 3: transaction: True, $1 placeholders: True, calls db.transaction: True


With the rule in the context, all three answers ran both deletes in a transaction and sent them
with `$1` placeholders. Only one called `db.transaction` by name, while the other two wrote `BEGIN`
and `COMMIT` by hand, so the rule made the safe shape reliable without making the agent use the
helper every time.

| Matcher | Rule selected for src/api/db.ts | Answers with a transaction |
|---|---|---|
| `fnmatch` | none | 1 of 3 |
| `match_path_to_glob` | `api-database.md` | 3 of 3 |

## Step 6: Write each instruction in exactly one file

A task that touches both folders loads both rules, and each rule carries lines it copied from the
instruction file. We first find every instruction line that is written in more than one file, and
measure what those copies cost with the provider's own count.

![Write each instruction in exactly one file](images/context-budget-step-1.svg)

In [14]:
def find_repeated_instructions():
    """Map each instruction line written in more than one file to the files that hold it."""
    holders = {}
    for path in [INSTRUCTION_PATH, *sorted((WORKSPACE / "rules").glob("*.md"))]:
        for line in path.read_text().splitlines():
            if line.startswith("- "):
                holders.setdefault(line, []).append(path.name)
    return {line: names for line, names in holders.items() if len(names) > 1}


for line, names in find_repeated_instructions().items():
    print(f"{len(names)} copies in {names}:\n    {line}")

3 copies in ['AGENTS.md', 'api-database.md', 'ui-components.md']:
    - Use pnpm for every package command, never npm or yarn.
2 copies in ['AGENTS.md', 'api-database.md']:
    - Run pnpm test before you commit.


`count_prompt_tokens` asks the provider how many prompt tokens a piece of text costs, by sending it
with a one-token reply budget and reading `usage.prompt_tokens` from the response.

In [15]:
def count_prompt_tokens(text):
    """Prompt tokens the provider charges for this text, read from a one-token reply."""
    response = client.chat.completions.create(
        model=MODEL, max_tokens=1, messages=[{"role": "user", "content": text}])
    return response.usage.prompt_tokens


FULL_STACK_PATHS = ["src/ui/components/OrderList.tsx", "src/api/routes/orders.ts"]
full_stack_context, selected = assemble_scoped_context(FULL_STACK_PATHS, match_path_to_glob)
tokens_with_copies = count_prompt_tokens(full_stack_context)
print(f"rules selected: {selected}")
print(f"full-stack context with the copies: {tokens_with_copies} prompt tokens")

rules selected: ['api-database.md', 'ui-components.md']
full-stack context with the copies: 205 prompt tokens


The instruction file loads on every task, so a rule never needs to repeat it. The next cell deletes
those lines from the rule files, which is the edit you would make by hand, and reads the rules again.

In [16]:
def delete_repeated_instructions():
    """Remove from every rule file each line that the instruction file already holds."""
    shared = {line for line in INSTRUCTION_PATH.read_text().splitlines() if line.startswith("- ")}
    for path in sorted((WORKSPACE / "rules").glob("*.md")):
        kept = [line for line in path.read_text().splitlines() if line not in shared]
        path.write_text("\n".join(kept) + "\n")


delete_repeated_instructions()
RULES = read_all_rules()
full_stack_context, _ = assemble_scoped_context(FULL_STACK_PATHS, match_path_to_glob)
tokens_without_copies = count_prompt_tokens(full_stack_context)
print(f"repeated instructions left: {len(find_repeated_instructions())}")
print(f"with the copies    : {tokens_with_copies} prompt tokens")
print(f"without the copies : {tokens_without_copies} prompt tokens")

repeated instructions left: 0
with the copies    : 205 prompt tokens
without the copies : 165 prompt tokens


## Step 7: Refuse any context that goes over the budget

Nothing so far stops the context from growing, so we give it a **context budget**, the most prompt
tokens we are willing to spend on instructions for one task. We choose that number ourselves, far
below the model's **context window**, which is the total amount of text a model can hold in one
request.

![Refuse any context that goes over the budget](images/context-budget-step-2.svg)

In [17]:
CONTEXT_BUDGET_TOKENS = 300   # our own ceiling for the instructions on one task


class ContextBudgetError(Exception):
    """The assembled context is over budget, so the request is never sent."""


def assemble_context_within_budget(changed_paths, count_tokens):
    """Assemble the scoped context, and refuse it if it costs more than the budget."""
    context, selected = assemble_scoped_context(changed_paths, match_path_to_glob)
    used = count_tokens(context)
    if used > CONTEXT_BUDGET_TOKENS:
        sizes = {rule["name"]: count_tokens(rule["body"])
                 for rule in RULES if rule["name"] in selected}
        raise ContextBudgetError(f"{used} tokens is over the budget of "
                                 f"{CONTEXT_BUDGET_TOKENS}. Rule sizes: {sizes}")
    return context, used


window = provider_truth()["models"][MODEL]["context_length"]
print(f"budget {CONTEXT_BUDGET_TOKENS} tokens, context window {window} tokens")

budget 300 tokens, context window 1048576 tokens


Every task we have so far fits. Then a teammate adds a rule for database migrations, pastes the
whole migration history into it, and scopes it to `src/**` so that it never gets missed.

In [18]:
for paths in (["src/ui/components/OrderList.tsx"], ["src/api/db.ts"], FULL_STACK_PATHS):
    _, used = assemble_context_within_budget(paths, count_prompt_tokens)
    print(f"{used:4} tokens for {paths}")

migration_history = "\n".join(f"- Migration {number:04d} has already run, so never edit it."
                              for number in range(1, 41))
(WORKSPACE / "rules" / "api-migrations.md").write_text(
    f"---\npaths: src/**\n---\n# Database migrations\n{migration_history}\n")
RULES = read_all_rules()
print(f"\nadded rules/api-migrations.md, now {len(RULES)} rules")

 101 tokens for ['src/ui/components/OrderList.tsx']


 133 tokens for ['src/api/db.ts']


 165 tokens for ['src/ui/components/OrderList.tsx', 'src/api/routes/orders.ts']

added rules/api-migrations.md, now 3 rules


The frontend task goes through the assembler again, and this time the budget check has something
to say.

In [19]:
try:
    assemble_context_within_budget(["src/ui/components/OrderList.tsx"], count_prompt_tokens)
except ContextBudgetError as error:
    print(f"refused: {error}")

refused: 785 tokens is over the budget of 300. Rule sizes: {'api-migrations.md': 683, 'ui-components.md': 31}


The migration rule alone costs 683 tokens, and because `src/**` matches every file, the frontend
task would have paid 785 tokens for a change to one component. The error names the rule that is too
big, so the fix goes in that rule file. The next cell narrows its glob to the migrations folder and
replaces the history with the one instruction it was meant to carry.

In [20]:
(WORKSPACE / "rules" / "api-migrations.md").write_text(
    "---\npaths: src/api/migrations/**\n---\n# Database migrations\n"
    "- Never edit a migration that has already run. Add a new one instead.\n")
RULES = read_all_rules()

for paths in (["src/ui/components/OrderList.tsx"], ["src/api/migrations/0041_add_index.sql"]):
    _, used = assemble_context_within_budget(paths, count_prompt_tokens)
    print(f"{used:4} tokens for {paths}")

 101 tokens for ['src/ui/components/OrderList.tsx']


  90 tokens for ['src/api/migrations/0041_add_index.sql']


## Step 8: Summarise the conversation but never the rules

A long coding session outgrows any budget, so agents shrink it with **compaction**, which means
shrinking a long conversation so it still fits and still makes sense. The first version most agents
try summarises the whole request, rules included, and keeps only the last two turns word for word.

![Summarise the conversation but never the rules](images/context-budget-step-3.svg)

In [21]:
SESSION_TURNS = [
    "user: The orders endpoint in src/api/routes/orders.ts is slow on large accounts.",
    "assistant: It loads every order and filters in memory. I will add a WHERE clause.",
    "user: Good. Also return the orders 50 at a time.",
    "assistant: Added a created_at cursor to the query and an index migration for it.",
    "user: The endpoint tests fail on an account with no orders.",
    "assistant: Fixed. An empty page now returns an empty list and no cursor.",
    "user: Next, move orders older than two years from orders into orders_archive.",
]


def summarise_text(text):
    """Rewrite a long session as a short summary, with the small model."""
    response = client.chat.completions.create(
        model=SUMMARY_MODEL, max_tokens=250,
        messages=[{"role": "system", "content": "Summarise this coding session in under 100 words."},
                  {"role": "user", "content": text}])
    return response.choices[0].message.content

`compact_whole_session` is that first version. The last turn asks for a write to two tables, which
is exactly the case the database rule exists for.

In [22]:
def compact_whole_session(context, turns, summarise, keep_last=2):
    """The first version: summarise the rules and the older turns together."""
    older, recent = turns[:-keep_last], turns[-keep_last:]
    summary = summarise(context + "\n\n" + "\n".join(older))
    return "Summary of the session so far:\n" + summary, "\n".join(recent)


orders_context, _ = assemble_context_within_budget(["src/api/routes/orders.ts"],
                                                   count_prompt_tokens)
for attempt in range(1, 4):
    system, task = compact_whole_session(orders_context, SESSION_TURNS, summarise_text)
    answer, _ = ask_coding_agent(system, task)
    print(f"attempt {attempt}: summary keeps db.transaction: {'db.transaction' in system}, "
          f"answer uses a transaction: {wraps_writes_in_transaction(answer)}")

attempt 1: summary keeps db.transaction: False, answer uses a transaction: True


attempt 2: summary keeps db.transaction: False, answer uses a transaction: False


attempt 3: summary keeps db.transaction: False, answer uses a transaction: False


None of the three summaries kept `db.transaction` by name, and each one shrank the database rules
to a short phrase about transactions and placeholders. With only that phrase to go on, one of the
three answers put the archive move in a transaction, while the other two wrote a general plan or
repeated the summary back.

The fix treats the instruction file and the rules as **persistent facts**, which are read fresh
from disk on every turn and never handed to the summariser. `build_compacted_request` summarises
only the older conversation turns, then puts the scoped context back in front of the summary.

In [23]:
def build_compacted_request(changed_paths, turns, summarise, count_tokens, keep_last=2):
    """Rules are read from disk on every turn. Only the conversation is summarised."""
    context, _ = assemble_context_within_budget(changed_paths, count_tokens)
    older, recent = turns[:-keep_last], turns[-keep_last:]
    task = ("Summary of earlier turns:\n" + summarise("\n".join(older))
            + "\n\nLast turns, word for word:\n" + "\n".join(recent))
    return context, task


for attempt in range(1, 4):
    system, task = build_compacted_request(["src/api/routes/orders.ts"], SESSION_TURNS,
                                           summarise_text, count_prompt_tokens)
    answer, _ = ask_coding_agent(system, task)
    print(f"attempt {attempt}: rules in the request: {'db.transaction' in system}, "
          f"answer uses a transaction: {wraps_writes_in_transaction(answer)}")

attempt 1: rules in the request: True, answer uses a transaction: False


attempt 2: rules in the request: True, answer uses a transaction: True


attempt 3: rules in the request: True, answer uses a transaction: True


The rules reached the model on all three turns, because they never passed through the summariser.
Two of the three answers put the archive move in a transaction, one of them through `db.transaction`
after running `pnpm db:check`, and the third went back to the cursor work from an earlier turn.
Having a rule in the request does not guarantee the model follows it, but a rule lost in a summary
cannot be followed at all.

## Step 9: Test the context assembler without calling the model

Each fix above gets a test that runs in milliseconds with no API key, so it can run on every commit.
The tests count words instead of calling the provider, because a budget test only needs a counter
that grows with the text.

![Test the context assembler without calling the model](images/context-budget-step-4.svg)

In [24]:
def count_words(text):
    """A stand-in token counter for tests, so they never call the provider."""
    return len(text.split())


def test_frontend_change_loads_no_database_rule():
    _, selected = assemble_scoped_context(["src/ui/components/OrderList.tsx"], match_path_to_glob)
    assert selected == ["ui-components.md"], selected


def test_root_backend_file_loads_database_rule():
    _, selected = assemble_scoped_context(["src/api/db.ts"], match_path_to_glob)
    assert "api-database.md" in selected, selected


def test_no_instruction_is_written_twice():
    assert find_repeated_instructions() == {}, find_repeated_instructions()

The last three tests cover the budget, a rule with no scope, and a summary that throws everything
away.

In [25]:
def test_budget_refuses_an_oversized_rule():
    RULES.append({"name": "huge.md", "glob": "src/**", "body": "- one more note\n" * 500})
    try:
        assemble_context_within_budget(["src/ui/App.tsx"], count_words)
    except ContextBudgetError:
        return
    finally:
        RULES.pop()
    raise AssertionError("a context over the budget was allowed through")


def test_rule_without_scope_is_refused():
    unscoped = WORKSPACE / "unscoped-rule.md"
    unscoped.write_text("- Always run the linter.\n")
    try:
        read_rule_file(unscoped)
    except ValueError:
        return
    finally:
        unscoped.unlink()
    raise AssertionError("a rule with no paths: line was loaded")

The compaction test hands the assembler a summariser that returns nothing at all, and still expects
the database rule in the request.

In [26]:
def test_rules_survive_an_empty_summary():
    system, _ = build_compacted_request(["src/api/db.ts"], SESSION_TURNS,
                                        lambda text: "", count_words)
    assert "db.transaction" in system, "the database rule was lost in compaction"


for test in (test_frontend_change_loads_no_database_rule, test_root_backend_file_loads_database_rule,
             test_no_instruction_is_written_twice, test_budget_refuses_an_oversized_rule,
             test_rule_without_scope_is_refused, test_rules_survive_an_empty_summary):
    test()
    print(f"passed: {test.__name__}")
shutil.rmtree(WORKSPACE)
print("temporary monorepo removed")

passed: test_frontend_change_loads_no_database_rule
passed: test_root_backend_file_loads_database_rule
passed: test_no_instruction_is_written_twice
passed: test_budget_refuses_an_oversized_rule
passed: test_rule_without_scope_is_refused
passed: test_rules_survive_an_empty_summary
temporary monorepo removed


## Concepts

| Concept | Where it lives | What it does |
|---|---|---|
| **Instruction file** | `AGENTS.md`, read by `assemble_scoped_context` | Holds the settings that apply to every task, and loads every time |
| **Path-scoped rule** | a rule file with a `paths:` line, read by `read_rule_file` | Loads only when the task changes a path its glob matches |
| **Glob matching** | `match_path_to_glob` | Reads `**/` as any number of folders, which `fnmatch` does not |
| **One copy of each instruction** | `find_repeated_instructions` | Finds a line written in more than one file, so it is paid for once |
| **Context budget** | `assemble_context_within_budget` | Refuses to send a context over the budget, and names the rules that are too big |
| **Compaction** | `build_compacted_request` | Summarises the conversation and reads the rules fresh from disk every turn |